In [ ]:
# Install Groq library
# To connect our application with Groq's LLM API.
!pip install groq -q

from groq import Groq
from google.colab import userdata

# Read API key from Colab Secrets
# Creates a connection to the Groq API using the stored API key. All LLM requests will be sent through this client.
groq_client = Groq(api_key=userdata.get("GROQ_KEY"))

response = groq_client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": "What is RAG in AI?"}
    ]
)

print(response.choices[0].message.content)

RAG (Retrieval-Augmented Generation) is a type of artificial intelligence (AI) model that combines the strengths of retrieval-based and generation-based approaches to produce more accurate and informative text outputs.

In traditional generation-based models, the AI generates text from scratch based on a given prompt or input. However, this approach can lead to issues such as:

1. Lack of context: The model may not have access to relevant information or context that is necessary to generate accurate and informative text.
2. Hallucinations: The model may generate text that is not based on actual facts or data, but rather on patterns and associations learned during training.

RAG models address these issues by augmenting the generation process with a retrieval step. The retrieval step involves searching a large database or knowledge graph to retrieve relevant information related to the input prompt or query. This retrieved information is then used to inform and guide the generation proce

In [ ]:
!pip install sentence-transformers scikit-learn -q

In [ ]:
from google import genai
from google.colab import userdata
import numpy as np

try:
    # Get API key
    api_key = userdata.get("GEMINI_KEY")

    if not api_key:
        raise ValueError("GEMINI_KEY not found in Colab Secrets.")

    client = genai.Client(api_key=api_key)

    sentences = [
        "Python is one of the most popular programming languages for AI development.",
        "Machine learning models learn patterns from large datasets.",
        "Cricket is the most widely followed sport in India.",
        "Football clubs spend millions on player transfers every season.",
        "Data science combines statistics, programming, and domain knowledge.",
        "Deep learning uses neural networks with many hidden layers.",
        "The Indian Space Research Organisation launches satellites into orbit.",
        "Basketball players require excellent speed and coordination.",
        "Cloud computing provides scalable resources over the internet.",
        "Cybersecurity protects systems from unauthorized access and attacks.",
        "Artificial intelligence is transforming healthcare diagnostics.",
        "Natural language processing enables computers to understand human language.",
        "Tennis matches can last several hours in major tournaments.",
        "Big data technologies help companies analyze massive datasets.",
        "Generative AI can create text, images, and code.",
        "Swimming is a full-body workout that improves endurance.",
        "Databases store and organize information efficiently.",
        "The Olympic Games bring together athletes from around the world.",
        "Prompt engineering improves the quality of AI-generated responses.",
        "Software developers use version control systems like Git."
    ]

    print("Embedding text strings into vector space...")

    response = client.models.embed_content(
        model="gemini-embedding-001",
        contents=sentences
    )

    if not response.embeddings:
        raise ValueError("No embeddings returned for sentences.")

    vectors = [embedding.values for embedding in response.embeddings]

    query = "How is artificial intelligence used in modern technology?"

    query_response = client.models.embed_content(
        model="gemini-embedding-001",
        contents=query
    )

    if not query_response.embeddings:
        raise ValueError("No embedding returned for query.")

    query_embedding = query_response.embeddings[0].values

    # Cosine similarity measures semantic closeness
    def cosine_similarity(v1, v2):
        denominator = np.linalg.norm(v1) * np.linalg.norm(v2)

        if denominator == 0:
            return 0.0

        return np.dot(v1, v2) / denominator

    scores = [cosine_similarity(query_embedding, v) for v in vectors]

    top_indices = np.argsort(scores)[::-1][:3]

    print(f"\nQuery: '{query}'\n")
    print("TOP 3 SEMANTIC MATCHES:")

    for rank, idx in enumerate(top_indices, 1):
        print(f"{rank}. [Score: {scores[idx]:.4f}] {sentences[idx]}")

except ValueError as ve:
    print(f"Value Error: {ve}")

except KeyError as ke:
    print(f"Missing Key: {ke}")

except ConnectionError as ce:
    print(f"Connection Error: {ce}")

except Exception as e:
    print(f"Unexpected Error: {type(e).__name__}: {e}")

Embedding text strings into vector space...

Query: 'How is artificial intelligence used in modern technology?'

TOP 3 SEMANTIC MATCHES:
1. [Score: 0.6734] Artificial intelligence is transforming healthcare diagnostics.
2. [Score: 0.6542] Python is one of the most popular programming languages for AI development.
3. [Score: 0.6084] Machine learning models learn patterns from large datasets.


In [ ]:
!pip install groq -q

from groq import Groq
from google.colab import userdata

groq_client = Groq(api_key=userdata.get("GROQ_KEY"))

# Then use a LLM from GROQ to make the final OUTPUT

retrieved_context = "\n".join([
    f"- {sentences[idx]}" for idx in top_indices
])

system_prompt = (
    "You are a helpful AI assistant. Answer the user's question strictly "
    "using the provided context. If the answer is not present in the context, "
    "say 'The provided context does not contain enough information to answer this question.'"
)

augmented_user_prompt = f"""
Context:
{retrieved_context}

User Question: {query}

Answer:
"""

groq_response = groq_client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": augmented_user_prompt}
    ]
)

print("\nRETRIEVED CONTEXT:")
print(retrieved_context)

print("\nFINAL RAG ANSWER FROM GROQ:")
print(groq_response.choices[0].message.content)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.0 MB/s eta 0:00:00

RETRIEVED CONTEXT:
- Artificial intelligence is transforming healthcare diagnostics.
- Python is one of the most popular programming languages for AI development.
- Machine learning models learn patterns from large datasets.

FINAL RAG ANSWER FROM GROQ:
Artificial intelligence is transforming healthcare diagnostics. Additionally, it is used in AI development, with Python being one of the most popular programming languages for this purpose. Machine learning models, a part of artificial intelligence, learn patterns from large datasets.
